<a href="https://colab.research.google.com/github/kee-tech/Crowd-Surveillance-and-Panic-Detection/blob/main/RAG_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: Building the Retrieval Pipeline for RAG

## Objective

In this lab, we build the **Retrieval component** of a Retrieval-Augmented Generation (RAG) system.

We will:

1. Load documents
2. Split documents into chunks
3. Generate embeddings
4. Store embeddings in a vector database
5. Retrieve relevant document chunks for a user query

## Technology Stack

- Python
- LangChain
- Hugging Face Sentence Transformers
- ChromaDB

> This lab focuses only on ingestion and retrieval. LLM-based answer generation can be added in the next lab.


## Step 1: Install Required Libraries

In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-chroma \
langchain-text-splitters \
chromadb \
sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

## Step 2: Import Required Libraries

LangChain provides reusable components for document processing, chunking, embeddings, and retrieval.

In [2]:
from pathlib import Path
import shutil

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_1666/2997991053.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


## Step 3: Configure the Retrieval Pipeline

In [3]:
DATA_DIR = Path("data")
VECTOR_DB_DIR = Path("vector_db")

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 3

DATA_DIR.mkdir(exist_ok=True)

print("Configuration loaded successfully.")

Configuration loaded successfully.


# Step 4: Create Sample Documents

For this first experiment, we create a small knowledge base.

In [4]:
sample_documents = {
    "rag.txt": """
Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved information can later be provided to a language
model as context.

RAG helps connect language models with external knowledge
sources.
""",

    "vector_database.txt": """
A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user submits a query, the query is also converted into
an embedding.

Similarity search is then used to identify document chunks
that are semantically similar to the query.
""",

    "chunking.txt": """
Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information between neighboring chunks.

Effective chunking can improve retrieval quality because
the retriever searches meaningful pieces of information.
"""
}

## Step 5: Save the Documents

In [5]:
for filename, content in sample_documents.items():
    file_path = DATA_DIR / filename
    file_path.write_text(content.strip(), encoding="utf-8")

print("Sample documents created:")
for file_path in DATA_DIR.glob("*.txt"):
    print("-", file_path.name)

Sample documents created:
- rag.txt
- chunking.txt
- vector_database.txt


# Step 6: Load Documents

Each document is converted into a LangChain `Document` object.

A document contains:
- `page_content` → actual text
- `metadata` → additional information such as the source file


In [6]:
documents = []

for file_path in DATA_DIR.glob("*.txt"):
    text = file_path.read_text(encoding="utf-8")

    document = Document(
        page_content=text,
        metadata={"source": file_path.name}
    )

    documents.append(document)

print(f"Loaded {len(documents)} documents.")

Loaded 3 documents.


## Display Loaded Documents

In [7]:
for document in documents:
    print("=" * 60)
    print("SOURCE:", document.metadata["source"])
    print()
    print(document.page_content)
    print()

SOURCE: rag.txt

Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved information can later be provided to a language
model as context.

RAG helps connect language models with external knowledge
sources.

SOURCE: chunking.txt

Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information between neighboring chunks.

Effective chunking can improve retrieval quality because
the retriever searches meaningful pieces of information.

SOURCE: vector_database.txt

A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user submits a query, the query is also converted into
an embedding.

Similarity search is then used to ident

# Step 7: Split Documents into Chunks

Large documents are divided into smaller chunks so that the retrieval system can find the most relevant parts of a document.

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

In [9]:
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 3 chunks.


## Display Chunks

In [10]:
for i, chunk in enumerate(chunks, start=1):
    print("=" * 60)
    print(f"CHUNK {i}")
    print()
    print("SOURCE:", chunk.metadata["source"])
    print()
    print(chunk.page_content)
    print()

CHUNK 1

SOURCE: rag.txt

Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved information can later be provided to a language
model as context.

RAG helps connect language models with external knowledge
sources.

CHUNK 2

SOURCE: chunking.txt

Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information between neighboring chunks.

Effective chunking can improve retrieval quality because
the retriever searches meaningful pieces of information.

CHUNK 3

SOURCE: vector_database.txt

A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user submits a query, the query is also converted into
an embedding.

Similarity s

# Step 8: Load the Embedding Model

Embeddings convert text into numerical vectors.

We use the free Hugging Face model:

`sentence-transformers/all-MiniLM-L6-v2`


In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

/tmp/ipykernel_1666/2755281263.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


## Test the Embedding Model

In [12]:
test_query = "What is Retrieval-Augmented Generation?"

embedding_vector = embeddings.embed_query(test_query)

print("Embedding vector dimension:")
print(len(embedding_vector))

Embedding vector dimension:
384


# Step 9: Create the Vector Database

Each chunk is converted into an embedding and stored in ChromaDB.

The vector database allows us to perform similarity search.

In [13]:
if VECTOR_DB_DIR.exists():
    shutil.rmtree(VECTOR_DB_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(VECTOR_DB_DIR),
    collection_name="rag_retrieval_lab"
)

print("Vector database created successfully.")

Vector database created successfully.


# Step 10: Create the Retriever

For a query:

1. The query is converted into an embedding
2. ChromaDB compares it with stored vectors
3. The most similar chunks are returned


In [14]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print("Retriever created successfully.")

Retriever created successfully.


# Step 11: Perform Semantic Retrieval

In [15]:
question = "How does RAG use external knowledge?"

retrieved_documents = retriever.invoke(question)

## Display Retrieval Results

In [16]:
print("QUESTION:")
print(question)

print()
print("=" * 60)
print("RETRIEVED DOCUMENT CHUNKS")

for i, document in enumerate(retrieved_documents, start=1):
    print()
    print("=" * 60)
    print(f"RESULT {i}")
    print()
    print("SOURCE:", document.metadata.get("source"))
    print()
    print(document.page_content)

QUESTION:
How does RAG use external knowledge?

RETRIEVED DOCUMENT CHUNKS

RESULT 1

SOURCE: rag.txt

Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved information can later be provided to a language
model as context.

RAG helps connect language models with external knowledge
sources.

RESULT 2

SOURCE: chunking.txt

Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information between neighboring chunks.

Effective chunking can improve retrieval quality because
the retriever searches meaningful pieces of information.

RESULT 3

SOURCE: vector_database.txt

A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user

# Step 12: Try Different Queries

In [17]:
questions = [
    "What are embeddings?",
    "Why is chunking important?",
    "What is stored in a vector database?",
    "How does RAG retrieve information?"
]

for question in questions:
    print()
    print("=" * 70)
    print("QUESTION:")
    print(question)

    retrieved_documents = retriever.invoke(question)

    print()
    print("RETRIEVED RESULTS:")

    for i, document in enumerate(retrieved_documents, start=1):
        print()
        print(f"Result {i}")
        print("Source:", document.metadata.get("source"))
        print(document.page_content)


QUESTION:
What are embeddings?

RETRIEVED RESULTS:

Result 1
Source: vector_database.txt
A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user submits a query, the query is also converted into
an embedding.

Similarity search is then used to identify document chunks
that are semantically similar to the query.

Result 2
Source: rag.txt
Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved information can later be provided to a language
model as context.

RAG helps connect language models with external knowledge
sources.

Result 3
Source: chunking.txt
Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information be

# Step 13: Create a Reusable Retrieval Function

In [18]:
def retrieve_documents(question, k=3):

    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    results = retriever.invoke(question)

    return results

## Test the Retrieval Function

In [19]:
question = "Explain the purpose of chunking."

results = retrieve_documents(question)

print("QUESTION:")
print(question)

for i, document in enumerate(results, start=1):
    print()
    print("=" * 60)
    print(f"RESULT {i}")
    print("SOURCE:", document.metadata.get("source"))
    print()
    print(document.page_content)

QUESTION:
Explain the purpose of chunking.

RESULT 1
SOURCE: chunking.txt

Chunking divides large documents into smaller pieces before
embedding and indexing.

Chunk size determines how much information is included in
each chunk.

Chunk overlap preserves information between neighboring chunks.

Effective chunking can improve retrieval quality because
the retriever searches meaningful pieces of information.

RESULT 2
SOURCE: vector_database.txt

A vector database stores numerical representations of
information called embeddings.

Embeddings represent the semantic meaning of text.

When a user submits a query, the query is also converted into
an embedding.

Similarity search is then used to identify document chunks
that are semantically similar to the query.

RESULT 3
SOURCE: rag.txt

Retrieval-Augmented Generation, commonly called RAG,
combines information retrieval with large language models.

A RAG system retrieves relevant information from an external
knowledge source.

The retrieved

# Lab Summary

In this lab, we implemented the **Retrieval component** of a RAG system.

## Pipeline Implemented

```text
Documents
    ↓
Document Loading
    ↓
Document Chunking
    ↓
Embedding Generation
    ↓
ChromaDB Vector Store
    ↓
User Query
    ↓
Query Embedding
    ↓
Similarity Search
    ↓
Top-K Relevant Chunks
```

## Important Concepts

### Documents
The knowledge source used by the RAG system.

### Chunking
Divides large documents into smaller meaningful pieces.

### Embeddings
Numerical vector representations that capture semantic meaning.

### Vector Database
Stores embeddings and performs similarity search.

### Retriever
Retrieves the most relevant chunks for a query.

## Next Lab

The next lab can extend this pipeline:

```text
Retrieved Chunks
        +
User Question
        ↓
Prompt Augmentation
        ↓
Large Language Model
        ↓
Grounded Answer
```
